# Runnable Configuration

The `config.py` module provides configuration types and utilities used to control Runnable execution, callbacks, tracing, recursion, concurrency, configurable fields, and context propagation.

## `EmptyDict`

An empty optional `TypedDict` used when a dictionary type with no predefined fields is required.

## `RunnableConfig`

A `TypedDict` that stores runtime configuration for a Runnable and its child calls. All fields are optional so configurations can be partially defined, inherited, and merged.

### Fields

1. `tags`: Stores tags for the current call and its child calls. Tags can be used to filter traced runs.
   * **Type:**
     ```python
     list[str]
     ```

2. `metadata`: Stores JSON-serializable metadata for the current call and its child calls.
   * **Type:**
     ```python
     dict[str, Any]
     ```

3. `callbacks`: Stores callbacks or a callback manager used for the current call and its child calls.
   * **Type:**
     ```python
     Callbacks
     ```

4. `run_name`: Specifies the name used for the tracer run.
   * **Type:**
     ```python
     str
     ```

5. `max_concurrency`: Specifies the maximum number of calls that may run in parallel.
   * **Type:**
     ```python
     int | None
     ```

6. `recursion_limit`: Specifies the maximum number of recursive Runnable calls. The default limit is `25`.
   * **Type:**
     ```python
     int
     ```

7. `configurable`: Stores runtime values for fields defined through `configurable_fields` or `configurable_alternatives`.
   * **Type:**
     ```python
     dict[str, Any]
     ```

8. `run_id`: Specifies the unique identifier used for the tracer run.
   * **Type:**
     ```python
     uuid.UUID | None
     ```

## Constants and Context Variables

1. `CONFIG_KEYS`: Lists the recognized top-level keys of a `RunnableConfig`.
   * **Value:**
     ```python
     [
         "tags",
         "metadata",
         "callbacks",
         "run_name",
         "max_concurrency",
         "recursion_limit",
         "configurable",
         "run_id",
     ]
     ```

2. `COPIABLE_KEYS`: Lists configuration fields that are copied when a configuration is inherited or normalized.
   * **Value:**
     ```python
     [
         "tags",
         "metadata",
         "callbacks",
         "configurable",
     ]
     ```

3. `CONFIGURABLE_TO_TRACING_METADATA_EXCLUDED_KEYS`: Stores configurable keys that must not automatically become tracing metadata.
   * **Value:**
     ```python
     frozenset(("api_key",))
     ```

4. `DEFAULT_RECURSION_LIMIT`: Stores the default maximum number of recursive Runnable calls.
   * **Value:**
     ```python
     25
     ```

5. `var_child_runnable_config`: Stores the configuration inherited by child Runnables within the current context.
   * **Type:**
     ```python
     ContextVar[RunnableConfig | None]
     ```

## Functions

1. `set_config_context`: Creates an isolated context containing the child Runnable configuration and its tracing context.
   * **Syntax:**
     ```python
     set_config_context(
         config: RunnableConfig # Configuration to set in the context
     ) -> Generator[Context, None, None]
     ```

2. `ensure_config`: Normalizes a configuration by adding default fields, inheriting the current child configuration, and moving unknown keys into `configurable`.
   * **Syntax:**
     ```python
     ensure_config(
         config: RunnableConfig | None = None # Configuration to normalize
     ) -> RunnableConfig
     ```

3. `get_config_list`: Converts one configuration or a sequence of configurations into a normalized list of the requested length.
   * **Syntax:**
     ```python
     get_config_list(
         config: RunnableConfig | Sequence[RunnableConfig] | None, # Configuration or configurations to normalize
         length: int # Required number of configurations
     ) -> list[RunnableConfig]
     ```

4. `patch_config`: Creates a normalized configuration and replaces selected callback, recursion, concurrency, naming, or configurable values.
   * **Syntax:**
     ```python
     patch_config(
         config: RunnableConfig | None, # Configuration to patch
         *,
         callbacks: BaseCallbackManager | None = None, # Callback manager to set
         recursion_limit: int | None = None, # Recursion limit to set
         max_concurrency: int | None = None, # Maximum concurrency to set
         run_name: str | None = None, # Run name to set
         configurable: dict[str, Any] | None = None # Configurable values to merge
     ) -> RunnableConfig
     ```

5. `merge_configs`: Merges multiple Runnable configurations into one configuration.
   * **Syntax:**
     ```python
     merge_configs(
         *configs: RunnableConfig | None # Configurations to merge
     ) -> RunnableConfig
     ```

6. `call_func_with_variable_args`: Calls a synchronous function and supplies `config` or `run_manager` only when the function accepts them.
   * **Syntax:**
     ```python
     call_func_with_variable_args(
         func: Callable[[Input], Output]
         | Callable[[Input, RunnableConfig], Output]
         | Callable[[Input, CallbackManagerForChainRun], Output]
         | Callable[
             [Input, CallbackManagerForChainRun, RunnableConfig],
             Output
         ], # Function to call
         input: Input, # Input passed to the function
         config: RunnableConfig, # Configuration passed when supported
         run_manager: CallbackManagerForChainRun | None = None, # Run manager passed when supported
         **kwargs: Any # Additional keyword arguments
     ) -> Output
     ```

7. `acall_func_with_variable_args`: Calls an asynchronous function and supplies `config` or `run_manager` only when the function accepts them.
   * **Syntax:**
     ```python
     acall_func_with_variable_args(
         func: Callable[[Input], Awaitable[Output]]
         | Callable[[Input, RunnableConfig], Awaitable[Output]]
         | Callable[
             [Input, AsyncCallbackManagerForChainRun],
             Awaitable[Output]
         ]
         | Callable[
             [Input, AsyncCallbackManagerForChainRun, RunnableConfig],
             Awaitable[Output]
         ], # Asynchronous function to call
         input: Input, # Input passed to the function
         config: RunnableConfig, # Configuration passed when supported
         run_manager: AsyncCallbackManagerForChainRun | None = None, # Run manager passed when supported
         **kwargs: Any # Additional keyword arguments
     ) -> Awaitable[Output]
     ```

8. `get_callback_manager_for_config`: Creates a synchronous callback manager from the callbacks, tags, and metadata stored in a Runnable configuration.
   * **Syntax:**
     ```python
     get_callback_manager_for_config(
         config: RunnableConfig # Configuration used to create the callback manager
     ) -> CallbackManager
     ```

9. `get_async_callback_manager_for_config`: Creates an asynchronous callback manager from the callbacks, tags, and metadata stored in a Runnable configuration.
   * **Syntax:**
     ```python
     get_async_callback_manager_for_config(
         config: RunnableConfig # Configuration used to create the callback manager
     ) -> AsyncCallbackManager
     ```

10. `get_executor_for_config`: Creates a context-aware thread-pool executor whose worker count is controlled by `max_concurrency`.
    * **Syntax:**
      ```python
      get_executor_for_config(
          config: RunnableConfig | None # Configuration containing the concurrency limit
      ) -> Generator[Executor, None, None]
      ```

11. `run_in_executor`: Asynchronously runs a synchronous function using a supplied executor or an executor selected from the Runnable configuration.
    * **Syntax:**
      ```python
      run_in_executor(
          executor_or_config: Executor | RunnableConfig | None, # Executor or configuration to use
          func: Callable[P, T], # Function to execute
          *args: P.args, # Positional arguments passed to the function
          **kwargs: P.kwargs # Keyword arguments passed to the function
      ) -> T
      ```

## `ContextThreadPoolExecutor`

A `ThreadPoolExecutor` that copies the current context into child threads so context variables, including Runnable configuration, remain available during threaded execution.

### Methods

1. `submit`: Submits a function to the thread pool while copying the current execution context into the worker thread.
   * **Syntax:**
     ```python
     submit(
         self,
         func: Callable[P, T], # Function to submit
         *args: P.args, # Positional arguments passed to the function
         **kwargs: P.kwargs # Keyword arguments passed to the function
     ) -> Future[T]
     ```

2. `map`: Applies a function to multiple iterables while providing each execution with a copied context.
   * **Syntax:**
     ```python
     map(
         self,
         fn: Callable[..., T], # Function to apply
         *iterables: Iterable[Any], # Iterables containing the function inputs
         **kwargs: Any # Executor mapping options such as timeout and chunksize
     ) -> Iterator[T]
     ```